In [1]:
import sys
from pathlib import Path

# Get parent directory (Thesis-Edvin)
sys.path.append(str(Path.cwd().parent))

In [2]:
from utils import *
from datasets import load_dataset
from rag import *

In [3]:
gpt_pred = load_dataset("Eathus/github-issues-vul-detection-gpt-few-results-final-best2", split="test")
gpt_pred_df = gpt_pred.to_pandas()
gpt_pred_df = gpt_pred_df[~gpt_pred_df.duplicated(subset="issue_github_id", keep=False)]


In [4]:
true_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
false_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & gpt_pred_df.cve_id.isna()]
false_neg = gpt_pred_df[~gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
all_true = gpt_pred_df[~gpt_pred_df.cve_id.isna()]

print("true pos:", len(true_pos))
print("false pos:", len(false_pos))
print("false neg:", len(false_neg))
print("all true:", len(all_true))
print("all", len(gpt_pred_df))

true pos: 296
false pos: 305
false neg: 10
all true: 306
all 1763


In [5]:
vc = true_pos['cve_primary_cwe'].value_counts()
len(vc)

109

In [6]:
cwes = load_dataset("Eathus/cwe_view1000_list_gpt_few_cwe_desc_final_best", split="train")
cwes_df = cwes.to_pandas()

# View CWE-1000

In [7]:
cwes_dict = cwes_df.to_dict('records')
documents_light = [
    create_ordered_cwe_document(weakness, old=True, doc_density=CWE_doc_density.LIGHT) for weakness in cwes_dict
]
documents_norm = [
    create_ordered_cwe_document(weakness, old=True, doc_density=CWE_doc_density.NORM) for weakness in cwes_dict
]
documents_heavy = [
    create_ordered_cwe_document(weakness, old=True, doc_density=CWE_doc_density.HEAVY) for weakness in cwes_dict
]

allowed_usage = ["Allowed", "Allowed-with-Review", "Discouraged"]
documents_light = filter_cwe_docs_usage(documents_light, allowed_usage)
documents_norm = filter_cwe_docs_usage(documents_norm, allowed_usage)
documents_heavy = filter_cwe_docs_usage(documents_heavy, allowed_usage)

documents_light = split_docs(documents_light, echo=True, max_tokens=800)
documents_norm = split_docs(documents_norm, echo=True, max_tokens=800)
documents_heavy = split_docs(documents_heavy, echo=True, max_tokens=800)

Original: 881 docs
After splitting: 886 docs
Original: 881 docs
After splitting: 933 docs
Original: 881 docs
After splitting: 975 docs


In [11]:
vectorstore_light = create_vectorstore(documents_light, "tmp/faiss_basic_index_light")
vectorstore_norm = create_vectorstore(documents_norm, "tmp/faiss_basic_index_norm")
vectorstore_heavy = create_vectorstore(documents_heavy, "tmp/faiss_basic_index_heavy")

Loading existing vector store...
Loading existing vector store...
Loading existing vector store...


In [20]:
ranges = [1, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore_light, match_col="gpt_description", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/basic_results/classification_report_light_top-{k}.md')

for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore_norm, match_col="gpt_description", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/basic_results/classification_report_norm_top-{k}.md')

for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore_heavy, match_col="gpt_description", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/basic_results/classification_report_heavy_top-{k}.md')


100%|██████████| 296/296 [00:05<00:00, 54.95it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report_light_top-1.md


100%|██████████| 296/296 [00:05<00:00, 52.75it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report_light_top-5.md


100%|██████████| 296/296 [00:05<00:00, 53.77it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report_light_top-10.md


... (20 outputs left out)

100%|██████████| 296/296 [00:06<00:00, 46.71it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report_heavy_top-15.md


100%|██████████| 296/296 [00:06<00:00, 45.92it/s]


✅ Markdown report saved to: tmp/basic_results/classification_report_heavy_top-20.md


# View CWE-1003

In [8]:
view1003 = load_dataset("Eathus/cwe_view1003_raw_list", split="train")
view1003_df = view1003.to_pandas()

In [9]:
documents1003_light = [doc for doc in documents_light if doc.metadata['CWE_ID'] in view1003_df['ID'].tolist()]
documents1003_norm = [doc for doc in documents_norm if doc.metadata['CWE_ID'] in view1003_df['ID'].tolist()]
documents1003_heavy = [doc for doc in documents_heavy if doc.metadata['CWE_ID'] in view1003_df['ID'].tolist()]

In [10]:
vectorstore1003_light = create_vectorstore(documents1003_light, "tmp/faiss_basic1003_index_light")
vectorstore1003_norm = create_vectorstore(documents1003_norm, "tmp/faiss_basic1003_index_norm")
vectorstore1003_heavy = create_vectorstore(documents1003_heavy, "tmp/faiss_basic1003_index_heavy")

Loading existing vector store...
Loading existing vector store...
Loading existing vector store...


In [11]:
ranges = [1, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore1003_light, match_col="gpt_description", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/basic_results/classification_report1003_light_top-{k}.md')

for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore1003_norm, match_col="gpt_description", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/basic_results/classification_report1003_norm_top-{k}.md')

for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore1003_heavy, match_col="gpt_description", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/basic_results/classification_report1003_heavy_top-{k}.md')

100%|██████████| 1264/1264 [00:23<00:00, 53.13it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report1003_light_top-1.md


100%|██████████| 1264/1264 [00:34<00:00, 36.94it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report1003_light_top-5.md


100%|██████████| 1264/1264 [00:46<00:00, 27.08it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report1003_light_top-10.md


... (20 outputs left out)

100%|██████████| 1264/1264 [00:54<00:00, 23.39it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report1003_heavy_top-15.md


100%|██████████| 1264/1264 [00:52<00:00, 24.06it/s]



✅ Markdown report saved to: tmp/basic_results/classification_report1003_heavy_top-20.md
